# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
%pip install -q requests beautifulsoup4 schedule pytest


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import json
import os
import re
import time
from datetime import datetime, timedelta

import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
BASE_URL = 'http://books.toscrape.com/catalogue/'
PAGE_URL = BASE_URL + 'page-{}.html'
DEFAULT_FILE = os.path.join('..', 'artifacts', 'books_data.txt')

# на сайте рейтинг задан словом в классе тега: <p class="star-rating Three">
RATINGS = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


def get_soup(url: str, session: requests.Session = None) -> BeautifulSoup:
    """Скачивает страницу и возвращает распарсенный HTML.

    Args:
        url (str): адрес страницы.
        session (requests.Session, optional): сессия для повторного
            использования соединения. Если не передана, запрос делается
            через обычный requests.get.

    Returns:
        BeautifulSoup: объект для поиска элементов на странице.

    Raises:
        requests.HTTPError: если сервер вернул код ошибки (например, 404).
    """
    http = session or requests
    response = http.get(url, timeout=10)
    response.raise_for_status()
    # сайт не указывает кодировку в заголовках, из-за этого
    # вместо "£" получается "Â£", поэтому задаём её явно
    response.encoding = 'utf-8'
    return BeautifulSoup(response.text, 'html.parser')


def get_book_data(book_url: str, session: requests.Session = None) -> dict:
    """Собирает информацию о книге с её страницы.

    Args:
        book_url (str): ссылка на страницу книги, например
            'http://books.toscrape.com/catalogue/
            a-light-in-the-attic_1000/index.html'.
        session (requests.Session, optional): сессия requests, чтобы
            не открывать новое соединение на каждую книгу.

    Returns:
        dict: словарь с данными о книге:
            - title (str): название;
            - category (str): жанр из "хлебных крошек";
            - price (float): цена в фунтах;
            - rating (int): рейтинг от 1 до 5 (0, если не найден);
            - in_stock (int): количество экземпляров в наличии;
            - description (str): описание (пустая строка, если его нет);
            - product_information (dict): таблица Product Information
              (UPC, Product Type, цены с налогом и без, и т.д.);
            - url (str): ссылка на страницу книги.
    """
    soup = get_soup(book_url, session)
    main = soup.find('div', class_='product_main')

    title = main.find('h1').get_text(strip=True)
    price = float(main.find('p', class_='price_color').get_text()[1:])

    rating_tag = main.find('p', class_='star-rating')
    rating = 0
    for cls in rating_tag['class']:
        if cls in RATINGS:
            rating = RATINGS[cls]

    # текст вида "In stock (22 available)" -> 22
    availability = main.find('p', class_='availability').get_text()
    numbers = re.findall(r'\d+', availability)
    in_stock = int(numbers[0]) if numbers else 0

    # описание лежит в первом <p> после блока с id="product_description",
    # у некоторых книг описания нет совсем
    description = ''
    desc_header = soup.find('div', id='product_description')
    if desc_header is not None:
        description = desc_header.find_next_sibling('p').get_text(strip=True)

    product_info = {}
    for row in soup.find('table', class_='table-striped').find_all('tr'):
        key = row.find('th').get_text(strip=True)
        value = row.find('td').get_text(strip=True)
        product_info[key] = value

    breadcrumb = soup.find('ul', class_='breadcrumb').find_all('a')
    category = breadcrumb[-1].get_text(strip=True)

    return {
        'title': title,
        'category': category,
        'price': price,
        'rating': rating,
        'in_stock': in_stock,
        'description': description,
        'product_information': product_info,
        'url': book_url,
    }

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'category': 'Poetry',
 'price': 51.77,
 'rating': 3,
 'in_stock': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def get_book_links(page_url: str, session: requests.Session = None) -> list:
    """Возвращает ссылки на все книги с одной страницы каталога.

    Args:
        page_url (str): адрес страницы каталога вида
            'http://books.toscrape.com/catalogue/page-1.html'.
        session (requests.Session, optional): сессия requests.

    Returns:
        list[str]: список абсолютных ссылок на страницы книг
            (на одной странице каталога 20 книг).
    """
    soup = get_soup(page_url, session)
    links = []
    for book in soup.find_all('article', class_='product_pod'):
        href = book.find('h3').find('a')['href']
        # ссылки относительные: 'a-light-in-the-attic_1000/index.html'
        links.append(BASE_URL + href)
    return links


def save_books(books: list, filename: str = DEFAULT_FILE) -> None:
    """Сохраняет список книг в текстовый файл, одна книга на строку (JSON).

    Args:
        books (list[dict]): данные о книгах.
        filename (str): путь к файлу. Папка создаётся, если её нет.
    """
    folder = os.path.dirname(filename)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(filename, 'w', encoding='utf-8') as f:
        for book in books:
            f.write(json.dumps(book, ensure_ascii=False) + '\n')


def scrape_books(is_save: bool = False, n_pages: int = None,
                 filename: str = DEFAULT_FILE) -> list:
    """Обходит страницы каталога и собирает данные обо всех книгах.

    Страницы перебираются по порядку (page-1.html, page-2.html, ...),
    пока сайт не вернёт ошибку 404 - это значит, что страницы закончились.

    Args:
        is_save (bool): если True, результат сохраняется в файл filename.
        n_pages (int, optional): сколько страниц обойти. По умолчанию
            None - обходятся все страницы (их 50, это 1000 книг).
        filename (str): путь к файлу для сохранения результата.

    Returns:
        list[dict]: список словарей с данными о книгах
            (формат словаря описан в get_book_data).
    """
    books = []
    page = 1
    with requests.Session() as session:
        while n_pages is None or page <= n_pages:
            try:
                links = get_book_links(PAGE_URL.format(page), session)
            except requests.HTTPError:
                break
            for link in links:
                books.append(get_book_data(link, session))
            print(f'Страница {page}: собрано {len(books)} книг')
            page += 1

    if is_save:
        save_books(books, filename)
        print(f'Данные сохранены в {filename}')
    return books

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True)
print(type(res), len(res))
print(res[-1]['title'], res[-1]['price'])

Страница 1: собрано 20 книг
Страница 2: собрано 40 книг
Страница 3: собрано 60 книг
Страница 4: собрано 80 книг
Страница 5: собрано 100 книг
Страница 6: собрано 120 книг
Страница 7: собрано 140 книг
Страница 8: собрано 160 книг
Страница 9: собрано 180 книг
Страница 10: собрано 200 книг
Страница 11: собрано 220 книг
Страница 12: собрано 240 книг
Страница 13: собрано 260 книг
Страница 14: собрано 280 книг
Страница 15: собрано 300 книг
Страница 16: собрано 320 книг
Страница 17: собрано 340 книг
Страница 18: собрано 360 книг
Страница 19: собрано 380 книг
Страница 20: собрано 400 книг
Страница 21: собрано 420 книг
Страница 22: собрано 440 книг
Страница 23: собрано 460 книг
Страница 24: собрано 480 книг
Страница 25: собрано 500 книг
Страница 26: собрано 520 книг
Страница 27: собрано 540 книг
Страница 28: собрано 560 книг
Страница 29: собрано 580 книг
Страница 30: собрано 600 книг
Страница 31: собрано 620 книг
Страница 32: собрано 640 книг
Страница 33: собрано 660 книг
Страница 34: собрано 68

In [7]:
# проверим, что файл записался и читается обратно
with open(DEFAULT_FILE, encoding='utf-8') as f:
    loaded = [json.loads(line) for line in f]
print(len(loaded))
loaded[0]['title'], loaded[0]['rating']

1000


('A Light in the Attic', 3)

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.

В `scraper.py` задача ставится на 19:00 каждый день, цикл бесконечный. Ниже для проверки запуск ставится на ближайшую минуту, и после первого выполнения цикл завершается, чтобы ячейка не висела вечно.

In [8]:
def job() -> None:
    """Задача для планировщика: собрать все книги и сохранить в файл."""
    print(time.strftime('%Y-%m-%d %H:%M:%S'), '- запуск сбора данных')
    scrape_books(is_save=True)


def run_schedule(run_time: str = '19:00', check_every: int = 60) -> None:
    """Запускает сбор данных каждый день в указанное время.

    Args:
        run_time (str): время запуска в формате 'чч:мм'.
        check_every (int): раз во сколько секунд проверять,
            не пора ли запускать задачу.
    """
    schedule.every().day.at(run_time).do(job)
    print(f'Сбор данных запланирован на {run_time} каждый день')
    while True:
        schedule.run_pending()
        time.sleep(check_every)

In [9]:
# проверка: запуск через минуту от текущего времени
run_time = (datetime.now() + timedelta(minutes=1)).strftime('%H:%M')
done = False


def test_job():
    global done
    job()
    done = True
    return schedule.CancelJob  # задача выполнится один раз


schedule.clear()
schedule.every().day.at(run_time).do(test_job)
print('Ждём запуска в', run_time)

while not done:
    schedule.run_pending()
    time.sleep(10)  # проверяем раз в 10 секунд, а не постоянно

print('Готово, задач в очереди:', len(schedule.get_jobs()))

Ждём запуска в 12:47
2026-11-16 12:47:07 - запуск сбора данных
Страница 1: собрано 20 книг
Страница 2: собрано 40 книг
Страница 3: собрано 60 книг
Страница 4: собрано 80 книг
Страница 5: собрано 100 книг
Страница 6: собрано 120 книг
Страница 7: собрано 140 книг
Страница 8: собрано 160 книг
Страница 9: собрано 180 книг
Страница 10: собрано 200 книг
Страница 11: собрано 220 книг
Страница 12: собрано 240 книг
Страница 13: собрано 260 книг
Страница 14: собрано 280 книг
Страница 15: собрано 300 книг
Страница 16: собрано 320 книг
Страница 17: собрано 340 книг
Страница 18: собрано 360 книг
Страница 19: собрано 380 книг
Страница 20: собрано 400 книг
Страница 21: собрано 420 книг
Страница 22: собрано 440 книг
Страница 23: собрано 460 книг
Страница 24: собрано 480 книг
Страница 25: собрано 500 книг
Страница 26: собрано 520 книг
Страница 27: собрано 540 книг
Страница 28: собрано 560 книг
Страница 29: собрано 580 книг
Страница 30: собрано 600 книг
Страница 31: собрано 620 книг
Страница 32: собрано

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [2]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! cd .. && pytest -q tests/test_scraper.py

......                                                                   [100%]
6 passed in 15.55s


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

**Ссылка на репозиторий:** ВСТАВИТЬ_ССЫЛКУ

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Нужно было написать парсер сайта Books to Scrape: собрать данные об одной книге, потом обо всех книгах каталога с сохранением в файл, настроить ежедневный запуск через schedule, написать тесты на pytest и оформить проект на GitHub. Получилось сделать задания 1–4: собрано 1000 книг, данные сохранены в books_data.txt, расписание срабатывает, все 6 тестов прошли. Задание 5 (GitHub) не сделал.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Сначала была ошибка: библиотека requests не находилась, хотя я её устанавливал в первой ячейке. Оказалось, что ! pip ставил библиотеки не в тот Python, на котором работал ноутбук. Заменил на %pip, перезапустил ядро, и всё заработало. Сбор всех 1000 книг занимает около 5 минут.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовал для подсмотра правильности струтуры переносов в коде`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: Как работать с Git и GitHub: ветки, коммиты, Pull Request. Как устроены регулярные выражения. Как правильно писать тесты для функций, которые обращаются к сайту.`

**5. Самооценка по критериям задания**:

`Ответ: Задания 1–4 выполнены полностью: 6,5 балла из 10. Задание 5 не выполнено.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Было полезно увидеть весь путь: от скачивания страницы до тестов. Сложнее всего показалось задание 5, если раньше не работал с Git, хорошо бы разобрать его на семинаре. В шаблоне опечатка: в ячейке с тестами написано test/ вместо tests/.`



